# 02 — Scan-on-scan: why a uniform sweep can be blind forever

The central analytical result. Two periodic processes do not intercept "with
probability *p* per look" — they either drift into coincidence or lock out
permanently.

Theory: [`docs/theory.md`](../docs/theory.md) §2.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from smartscan.analysis.scan_on_scan import (
    GOLDEN, analyse_coincidence, beam_dwell_s, poi_exponential,
    probability_of_intercept, three_distance_gaps,
)

TE, BEAMWIDTH, WR = 4.0, 2.0, 1e-3
WE = beam_dwell_s(BEAMWIDTH, TE)
print(f"emitter: Ts = {TE} s, beamwidth = {BEAMWIDTH} deg")
print(f"illumination window we = {WE * 1e3:.1f} ms per revolution ({100 * WE / TE:.2f}% duty)")

## The blindness table

`blind_fraction` is the proportion of initial emitter phases for which **no
sweep ever intercepts**, within a 600 s horizon.

In [ ]:
cases = [(0.096, "tuned sweep"), (0.5, "1/8"), (1.0, "1/4"), (2.0, "1/2"),
         (0.096 * GOLDEN, "golden-scaled")]
print(f"{'Tr (s)':>9}{'label':>16}{'Tr/Te':>10}{'rational':>10}{'commens':>9}{'blind':>8}{'E[TTI] (7)':>12}")
for tr, label in cases:
    r = analyse_coincidence(tr, TE, WR, WE, horizon_s=600.0)
    print(f"{tr:9.4f}{label:>16}{r.ratio:10.5f}{f'{r.rational[0]}/{r.rational[1]}':>10}"
          f"{str(r.commensurate):>9}{r.blind_fraction * 100:7.1f}%{r.closed_form_tti_s:12.1f}")
print()
print("The classical formula reports a finite mean for cases where ~99% of")
print("encounters never happen. It cannot represent blindness at all.")

## POI: staircase versus the exponential approximation

In [ ]:
t = np.linspace(0.05, 120.0, 700)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, (tr, label) in zip(axes, [(0.096, "Tr = 0.096 s (incommensurate)"),
                                  (2.0, "Tr = 2.0 s (commensurate, 1/2)")]):
    ax.plot(t, probability_of_intercept(tr, TE, WR, WE, t), lw=1.6, label="deterministic (true)")
    ax.plot(t, poi_exponential(tr, TE, WR, WE, t), "--", lw=1.2,
            label=r"$1-e^{-t/\tau}$ (common assumption)")
    ax.set_title(label); ax.set_xlabel("time (s)"); ax.grid(alpha=0.3); ax.legend()
    ax.set_ylim(-0.02, 1.02)
axes[0].set_ylabel("probability of intercept")
plt.tight_layout()

## The three-distance theorem

The golden ratio is the worst-approximable irrational, so it minimises the
largest gap in the sampled phase — the deterministic version of duty dithering.

In [ ]:
alphas = {"golden (1+sqrt5)/2": GOLDEN % 1, "pi": np.pi % 1, "sqrt2": np.sqrt(2) % 1,
          "1/10": 0.1, "1/4": 0.25, "1/2": 0.5}
print(f"{'alpha':>22}{'value':>10}{'distinct gaps':>15}{'largest gap':>14}")
for name, a in alphas.items():
    g = three_distance_gaps(a, 60)
    print(f"{name:>22}{a:10.6f}{len(g):15d}{g.max():14.5f}")

fig, ax = plt.subplots(figsize=(11, 2.6))
for i, (name, a) in enumerate(list(alphas.items())[:4]):
    pts = np.mod(np.arange(1, 61) * a, 1.0)
    ax.scatter(pts, np.full_like(pts, i), s=9, label=name)
ax.set_yticks(range(4)); ax.set_yticklabels(list(alphas)[:4])
ax.set_xlabel("phase, first 60 sweeps"); ax.set_title("Phase coverage by sweep-ratio choice")
plt.tight_layout()

## Estimating an unknown scan period from sparse intercepts

Two things must be right or the estimate is confidently wrong:
1. **Cluster** hits into beam arrivals — the raw series is a pulse train whose
   harmonics fool an argmax into reporting `Te/2` or `Te/4`.
2. **Deconvolve the sampling window** — otherwise the periodogram reports *our*
   sweep period.

In [ ]:
from smartscan.analysis.estimators import (
    cluster_arrivals, estimate_period_ls, estimate_period_sdif, spectral_window,
)

# A beam pass is a contiguous run of hits, not a single point.
rng = np.random.default_rng(0)
period, n_rev, dwell = 4000.0, 30, 60
hits = np.concatenate([
    np.arange(k * period, k * period + dwell) for k in range(n_rev)
    if rng.random() < 0.5  # we only catch about half the passes
])
print(f"{len(hits)} raw hit slots -> {len(cluster_arrivals(hits, 1000))} clustered arrivals")

for label, gap in [("clustered (correct)", None), ("raw pulse train", 0)]:
    est = estimate_period_ls(hits, period_min=1000, period_max=12000, cluster_gap=gap)
    err = abs(est.period - period) / period * 100 if est.valid else float("nan")
    print(f"  LS {label:22} -> {est.period:8.1f} slots  error {err:6.2f}%  conf {est.confidence:.2f}")
sd = estimate_period_sdif(hits, period_min=1000, period_max=12000)
print(f"  SDIF {'clustered':20} -> {sd.period:8.1f} slots  "
      f"error {abs(sd.period - period) / period * 100:6.2f}%")

## The spectral window: what a periodic sampler does to a periodogram

In [ ]:
visits = np.arange(0, 120000, 96.0)   # our own sweep, every 96 slots
periods = np.geomspace(50, 12000, 2000)
w = spectral_window(visits, periods)
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.semilogx(periods, w, lw=0.9)
ax.set_xlabel("candidate period (slots)"); ax.set_ylabel("window power W(f)")
ax.set_title("Spectral window of a 96-slot sweep: a comb at the period and every sub-multiple")
ax.grid(alpha=0.3)
plt.tight_layout()
print("Any apparent 'signal' at these periods is our own schedule. Deconvolving")
print("the window is what stops the estimator confidently reporting its own tail.")

## End-to-end validation against ground truth

Acceptance test 4. Runs on the 120 s config; see `docs/architecture.md` §17-A
for why 10 s cannot support a 2 % target.

In [ ]:
from smartscan.config import load_config
from smartscan.eval.scan_validation import validate_estimators

report = validate_estimators(load_config("../configs/scan_on_scan.yaml"), n_seeds=4)
for k, v in report["summary"].items():
    print(f"  {k:34} {v}")